In [0]:
silver_unified_path = "/Volumes/workspace/default/fleet_volume/silver_unified"

gold_base_df = spark.read.format("delta") \
    .load(silver_unified_path)

display(gold_base_df)

truck_id,event_time,latitude,longitude,speed,engine_temp,fuel_level
T10,2026-02-22T23:10:52.000Z,25.4514671286,81.8574675184,33.0,85.7775279797,53.5186548915
T1,2026-02-22T23:10:37.000Z,25.4272855587,81.8605934545,66.0,95.68274566,26.409700671
T3,2026-02-22T23:11:32.000Z,25.4469015458,81.8430492821,39.0,90.657208858,32.8965440899
T7,2026-02-22T23:11:27.000Z,25.3297926629,82.9601775113,47.0,87.9319534485,42.1429909207
T6,2026-02-22T23:11:47.000Z,25.3206557634,82.9541869921,42.0,79.9589133875,50.1108167215
T7,2026-02-22T23:11:17.000Z,26.8644467984,80.9553260906,45.0,87.9319534485,42.1429909207
T7,2026-02-22T23:11:57.000Z,25.323429903,82.9747346907,32.0,87.9319534485,42.1429909207
T7,2026-02-22T23:09:32.000Z,28.5193586107,77.4034537085,63.0,87.9319534485,42.1429909207
T10,2026-02-22T23:10:17.000Z,28.5289752512,77.377138567,30.0,85.7775279797,53.5186548915
T7,2026-02-22T23:10:27.000Z,28.5329982043,77.3943536966,79.0,87.9319534485,42.1429909207


#REAL-LIFE BUSINESS PROBLEMS + KPIs

#1️⃣ 🚨 Driver Safety Monitoring

Business Problem:
Accidents increase insurance cost and damage vehicles.

KPI: Risk Score per Truck

Combines:
Overspeed,
Engine overheating,
Low fuel (negligence)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count

gold_driver_safety = (
    gold_base_df
    .withColumn("overspeed_flag", (col("speed") > 70).cast("int"))
    .withColumn("overheat_flag", (col("engine_temp") > 95).cast("int"))
    .withColumn("low_fuel_flag", (col("fuel_level") < 20).cast("int"))
    .groupBy("truck_id")
    .agg(
        spark_sum("overspeed_flag").alias("overspeed_events"),
        spark_sum("overheat_flag").alias("overheat_events"),
        spark_sum("low_fuel_flag").alias("low_fuel_events"),
        count("*").alias("total_events")
    )
    .withColumn(
        "risk_score",
        col("overspeed_events") * 2 +
        col("overheat_events") * 3 +
        col("low_fuel_events") * 1
    )
)

display(gold_driver_safety)

truck_id,overspeed_events,overheat_events,low_fuel_events,total_events,risk_score
T7,2,0,0,15,4
T9,2,3,0,7,13
T4,3,3,0,3,15
T10,3,9,0,12,33
T1,2,3,0,5,13
T6,0,2,0,6,6
T5,0,2,0,12,6
T8,2,7,0,13,25
T2,0,1,0,2,3
T3,4,13,0,19,47


This solves insurance + compliance problem.

#2️⃣ 🛠 Preventive Maintenance KPI
Business Problem:
Breakdowns cost money and delay deliveries.

KPI: Engine Stress Index

Instead of just counting overheat events,
calculate average engine temp + spike frequency.

In [0]:
from pyspark.sql.functions import avg, max

gold_engine_health = (
    gold_base_df
    .groupBy("truck_id")
    .agg(
        avg("engine_temp").alias("avg_engine_temp"),
        max("engine_temp").alias("max_engine_temp")
    )
)

display(gold_engine_health)

truck_id,avg_engine_temp,max_engine_temp
T7,88.5328830275933,89.0586964093
T9,98.95033554785712,109.6309714981
T4,106.50115553453334,117.3220166266
T10,102.01459091842499,112.636144922
T1,96.94535059888,105.7488591714
T6,88.13804245146666,106.1681376424
T5,83.906495970875,107.714741149
T8,98.71739025177689,117.5650696078
T2,90.4153387797,106.2507227796
T3,96.01517248015266,112.6900181792


High avg temp → truck needs inspection.

#3️⃣ ⛽ Fuel Risk Monitoring
Business Problem:
Fuel theft & poor planning increase operational cost.

KPI: Fuel Drop Anomaly

Use window to detect sudden drops.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("truck_id").orderBy("event_time")

fuel_df = (
    gold_base_df
    .withColumn("prev_fuel", lag("fuel_level").over(window_spec))
    .withColumn("fuel_drop", col("prev_fuel") - col("fuel_level"))
)

gold_fuel_anomaly = (
    fuel_df
    .filter(col("fuel_drop") > 10)  # sudden drop
    .groupBy("truck_id")
    .count()
    .withColumnRenamed("count", "suspicious_fuel_drops")
)

display(gold_fuel_anomaly)

truck_id,suspicious_fuel_drops
T1,1
T10,3
T3,6
T4,1
T5,3
T6,2
T7,7
T8,5
T9,4


This mimics fuel theft detection.

#4️⃣ 🚛 Vehicle Utilization Score
💼 Business Problem:
Some trucks are underused while others are overloaded → uneven wear & revenue imbalance.

KPI: Active Usage kms per Truck


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col, radians, sin, cos, sqrt, atan2, sum as spark_sum

window_spec = Window.partitionBy("truck_id").orderBy("event_time")

distance_df = (
    gold_base_df
    .withColumn("prev_lat", lag("latitude").over(window_spec))
    .withColumn("prev_lon", lag("longitude").over(window_spec))
    .withColumn(
        "distance_km",
        6371 * 2 * atan2(
            sqrt(
                sin((radians(col("latitude") - col("prev_lat"))) / 2) ** 2 +
                cos(radians(col("prev_lat"))) *
                cos(radians(col("latitude"))) *
                sin((radians(col("longitude") - col("prev_lon"))) / 2) ** 2
            ),
            sqrt(1 - (
                sin((radians(col("latitude") - col("prev_lat"))) / 2) ** 2 +
                cos(radians(col("prev_lat"))) *
                cos(radians(col("latitude"))) *
                sin((radians(col("longitude") - col("prev_lon"))) / 2) ** 2
            ))
        )
    )
)

total_distance_df = (
    distance_df
    .groupBy("truck_id")
    .agg(
        spark_sum("distance_km").alias("total_distance_km")
    )
)

display(total_distance_df)

truck_id,total_distance_km
T1,1.3212496553006778
T10,1551.1720640366589
T2,0.0
T3,1089.0828886453232
T4,0.0
T5,1071.4928496710972
T6,263.9830386866393
T7,992.4246823489335
T8,781.9459064247798
T9,1120.8403174387427


In [0]:
# TIME UTILIZATION

# from pyspark.sql.window import Window
# from pyspark.sql.functions import lag, unix_timestamp, col, when, sum as spark_sum

# window_spec = Window.partitionBy("truck_id").orderBy("event_time")

# util_df = (
#     gold_base_df
#     .withColumn("prev_time", lag("event_time").over(window_spec))
#     .withColumn(
#         "time_diff_sec",
#         unix_timestamp("event_time") - unix_timestamp("prev_time")
#     )
#     .withColumn(
#         "active_time_sec",
#         when(col("speed") > 0, col("time_diff_sec")).otherwise(0)
#     )
# )

# gold_utilization_correct = (
#     util_df
#     .groupBy("truck_id")
#     .agg(
#         (spark_sum("active_time_sec") / 3600)
#         .alias("active_driving_hours")
#     )
# )

# display(gold_utilization_correct)

truck_id,active_driving_hours
T1,1.0
T10,1.9166666666666667
T2,0.0
T3,2.4166666666666665
T4,0.0
T5,3.25
T6,1.0833333333333333
T7,3.0
T8,2.4166666666666665
T9,1.3333333333333333


Very low hours → underutilized truck

Very high hours → overworked truck (maintenance risk)

#5️⃣ ⛽ Fuel Efficiency Indicator
💼 Business Problem:
Fuel is the biggest cost in logistics.

We approximate efficiency as:
Distance travelled / fuel consumed

Basic fuel consumption:

In [0]:
from pyspark.sql.functions import min, max

fuel_df = (
    gold_base_df
    .groupBy("truck_id")
    .agg(
        max("fuel_level").alias("max_fuel"),
        min("fuel_level").alias("min_fuel")
    )
    .withColumn(
        "fuel_consumed",
        col("max_fuel") - col("min_fuel")
    )
)

display(fuel_df)

truck_id,max_fuel,min_fuel,fuel_consumed
T7,42.1429909207,21.4106212726,20.7323696481
T9,92.9982621926,51.4823552685,41.515906924099994
T4,77.2355555182,20.0623421954,57.1732133228
T10,53.5186548915,24.3177144072,29.2009404843
T1,71.4391622177,26.409700671,45.029461546700006
T6,74.5627076821,49.6376639049,24.925043777199996
T5,84.6335714427,25.7436325466,58.889938896100006
T8,94.5824301681,46.9983129777,47.5841171904
T2,74.1147691023,63.6199312778,10.494837824500003
T3,89.9988816617,32.8965440899,57.1023375718


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_start = Window.partitionBy("truck_id").orderBy("event_time")
window_end = Window.partitionBy("truck_id").orderBy(col("event_time").desc())

fuel_correct = (
    gold_base_df
    .withColumn("rn_start", row_number().over(window_start))
    .withColumn("rn_end", row_number().over(window_end))
)

start_fuel = fuel_correct.filter(col("rn_start") == 1) \
    .select("truck_id", col("fuel_level").alias("start_fuel"))

end_fuel = fuel_correct.filter(col("rn_end") == 1) \
    .select("truck_id", col("fuel_level").alias("end_fuel"))

fuel_final = (
    start_fuel
    .join(end_fuel, "truck_id")
    .withColumn("fuel_consumed", col("start_fuel") - col("end_fuel"))
)

display(fuel_final)

truck_id,start_fuel,end_fuel,fuel_consumed
T1,39.7958516669,26.409700671,13.3861509959
T10,24.3177144072,53.5186548915,-29.2009404843
T2,63.6199312778,63.6199312778,0.0
T3,74.3885837569,32.8965440899,41.49203966700001
T4,62.4555600438,62.4555600438,0.0
T5,34.4826794162,78.921634648,-44.438955231799994
T6,74.5627076821,50.1108167215,24.451890960599997
T7,21.4106212726,42.1429909207,-20.7323696481
T8,46.9983129777,68.1132631089,-21.114950131199997
T9,77.5358413656,92.9982621926,-15.462420826999988


Window logic method is accurate, production safe, and handles basic  refuel.

In [0]:
efficiency_df = (
    total_distance_df
    .join(fuel_df, "truck_id")
    .withColumn(
        "fuel_efficiency_km_per_unit",
        col("total_distance_km") / col("fuel_consumed")
    )
)

display(efficiency_df)

truck_id,total_distance_km,max_fuel,min_fuel,fuel_consumed,fuel_efficiency_km_per_unit
T7,992.4246823489335,42.1429909207,21.4106212726,20.7323696481,47.86836715695369
T9,1120.8403174387427,92.9982621926,51.4823552685,41.515906924099994,26.99785216032688
T4,0.0,77.2355555182,20.0623421954,57.1732133228,0.0
T10,1551.1720640366589,53.5186548915,24.3177144072,29.2009404843,53.12062003176414
T1,1.3212496553006778,71.4391622177,26.409700671,45.029461546700006,0.029341893283143333
T6,263.9830386866393,74.5627076821,49.6376639049,24.925043777199996,10.59107623024983
T5,1071.4928496710972,84.6335714427,25.7436325466,58.889938896100006,18.19483717858055
T8,781.9459064247798,94.5824301681,46.9983129777,47.5841171904,16.432918221345837
T2,0.0,74.1147691023,63.6199312778,10.494837824500003,0.0
T3,1089.0828886453232,89.9988816617,32.8965440899,57.1023375718,19.07247469993535
